## 准备数据

In [1]:
import os
import numpy as np
import torch


def mnist_dataset(root='./data'):
    try:
        from torchvision import datasets
    except ImportError as e:
        raise ImportError('Please install torchvision: pip install torchvision') from e

    train_ds = datasets.MNIST(root=root, train=True, download=True)
    test_ds = datasets.MNIST(root=root, train=False, download=True)

    x = train_ds.data.numpy().astype(np.float32) / 255.0
    y = train_ds.targets.numpy().astype(np.int64)
    x_test = test_ds.data.numpy().astype(np.float32) / 255.0
    y_test = test_ds.targets.numpy().astype(np.int64)
    return (x, y), (x_test, y_test)


In [2]:
print(list(zip([1, 2, 3, 4], ['a', 'b', 'c', 'd'])))

[(1, 'a'), (2, 'b'), (3, 'c'), (4, 'd')]


## 建立模型

In [3]:
class myModel(torch.nn.Module):
    def __init__(self):
        super().__init__()
        ####################
        '''??????'''
        self.W1 = torch.nn.Parameter(torch.randn(28 * 28, 100) * 0.1)
        self.b1 = torch.nn.Parameter(torch.zeros(100))
        self.W2 = torch.nn.Parameter(torch.randn(100, 10) * 0.1)
        self.b2 = torch.nn.Parameter(torch.zeros(10))
        ####################

    def forward(self, x):
        ####################
        '''?????????logits'''
        x = x.reshape(-1, 28 * 28)
        h1 = torch.relu(x @ self.W1 + self.b1)
        logits = h1 @ self.W2 + self.b2
        ####################
        return logits


model = myModel()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)


## 计算 loss

In [4]:
def compute_loss(logits, labels):
    return torch.nn.functional.cross_entropy(logits, labels)


def compute_accuracy(logits, labels):
    predictions = torch.argmax(logits, dim=1)
    return torch.mean((predictions == labels).float())


def train_one_step(model, optimizer, x, y):
    model.train()
    optimizer.zero_grad()
    logits = model(x)
    loss = compute_loss(logits, y)
    loss.backward()
    optimizer.step()

    with torch.no_grad():
        accuracy = compute_accuracy(logits, y)
    return loss, accuracy


def test(model, x, y):
    model.eval()
    with torch.no_grad():
        logits = model(x)
        loss = compute_loss(logits, y)
        accuracy = compute_accuracy(logits, y)
    return loss, accuracy


## 实际训练

In [5]:
train_data, test_data = mnist_dataset()

x_train = torch.tensor(train_data[0], dtype=torch.float32)
y_train = torch.tensor(train_data[1], dtype=torch.long)
x_test = torch.tensor(test_data[0], dtype=torch.float32)
y_test = torch.tensor(test_data[1], dtype=torch.long)

# ???????????????????? mini-batch
for epoch in range(50):
    loss, accuracy = train_one_step(model, optimizer, x_train, y_train)
    print('epoch', epoch, ': loss', float(loss.item()), '; accuracy', float(accuracy.item()))

loss, accuracy = test(model, x_test, y_test)
print('test loss', float(loss.item()), '; accuracy', float(accuracy.item()))


100%|██████████| 9.91M/9.91M [06:06<00:00, 27.1kB/s]


epoch 0 : loss 2.4749412536621094 ; accuracy 0.066600002348423
epoch 1 : loss 2.3558335304260254 ; accuracy 0.1064833328127861
epoch 2 : loss 2.2492356300354004 ; accuracy 0.16113333404064178
epoch 3 : loss 2.1514415740966797 ; accuracy 0.2251666635274887
epoch 4 : loss 2.0598201751708984 ; accuracy 0.28913334012031555
epoch 5 : loss 1.9726999998092651 ; accuracy 0.3492000102996826
epoch 6 : loss 1.8889744281768799 ; accuracy 0.4092000126838684
epoch 7 : loss 1.808087706565857 ; accuracy 0.470466673374176
epoch 8 : loss 1.7297800779342651 ; accuracy 0.5295500159263611
epoch 9 : loss 1.6540205478668213 ; accuracy 0.5807166695594788
epoch 10 : loss 1.5808786153793335 ; accuracy 0.6223666667938232
epoch 11 : loss 1.5104856491088867 ; accuracy 0.6549666523933411
epoch 12 : loss 1.4428960084915161 ; accuracy 0.6795499920845032
epoch 13 : loss 1.3781743049621582 ; accuracy 0.701449990272522
epoch 14 : loss 1.3162845373153687 ; accuracy 0.7186166644096375
epoch 15 : loss 1.2571789026260376 ; 